# xRFM-Only Benchmark (CSV Only)

Pipeline: **Setup → Select Dataset → Load/Split → Train → Evaluate → Save**

- Uses `pytabkit` sklearn wrapper (handles preprocessing internally)
- CSV input only
- `RUN_MODE` switch: `"single"` or `"all"`
- Real-time ETA feedback

In [18]:
# ========== 1. SETUP ==========

import json, time, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                            r2_score, root_mean_squared_error)
from IPython.display import display

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

PROJECT_ROOT = Path.cwd()
OUTPUT_DIR   = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# ---------- USER SWITCHES ----------
RUN_MODE          = "single"              # "single" or "all"
SINGLE_DATASET_ID = "online_news_popularity"   # only used when RUN_MODE == "single"
TEST_SIZE         = 0.2
VAL_SIZE          = 0.2                # fraction of train+val used for val

print(f"Run mode: {RUN_MODE}")

Run mode: single


In [19]:
# ========== 1b. SETUP – pytabkit / xRFM ==========

try:
    from pytabkit import XRFM_D_Classifier, XRFM_D_Regressor
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pytabkit[models]"])
    from pytabkit import XRFM_D_Classifier, XRFM_D_Regressor

print("pytabkit xRFM ready.")

pytabkit xRFM ready.


In [20]:
# ========== 2. SELECT DATASET(S) ==========

with open(PROJECT_ROOT / "Datasets.json", encoding="utf-8") as f:
    CATALOG = {d["id"]: d for d in json.load(f)}

if RUN_MODE == "single":
    assert SINGLE_DATASET_ID in CATALOG, f"{SINGLE_DATASET_ID} not in Datasets.json"
    SELECTED = [SINGLE_DATASET_ID]
else:
    SELECTED = list(CATALOG.keys())

print(f"Datasets to run ({len(SELECTED)}): {SELECTED}")

Datasets to run (1): ['online_news_popularity']


In [21]:
# ========== 3. LOAD / SPLIT / PREPROCESS ==========
#
# We preprocess ourselves (StandardScaler + OneHotEncoder) and pass pure
# float32 numpy arrays to pytabkit.  This bypasses pytabkit's internal
# dtype conversion which has bugs with pandas 2.x StringDtype and with
# datasets that have zero categorical columns.

def load_and_split(cfg: dict):
    """Read CSV → clean → split → preprocess → return numpy arrays."""
    path = PROJECT_ROOT / cfg["csv_path"]
    assert str(path).lower().endswith(".csv"), "Only CSV files are accepted."

    df = pd.read_csv(path)

    # Normalize header names because some CSV files contain leading/trailing spaces.
    df.columns = [str(c).strip() for c in df.columns]

    # Resolve target column robustly (exact first, then strip/lower fallback).
    target_col = cfg["target"]
    if target_col not in df.columns:
        target_map = {str(c).strip().lower(): c for c in df.columns}
        mapped = target_map.get(str(target_col).strip().lower())
        if mapped is None:
            raise KeyError(f"Target column '{target_col}' not found in CSV: {path}")
        target_col = mapped

    # Resolve drop columns after header normalization.
    drop_cols = []
    col_map = {str(c).strip().lower(): c for c in df.columns}
    for c in cfg.get("drop_cols", []):
        mapped = col_map.get(str(c).strip().lower())
        if mapped is not None and mapped != target_col:
            drop_cols.append(mapped)
    if drop_cols:
        df = df.drop(columns=drop_cols)

    X = df.drop(columns=target_col)
    y = df[target_col]

    # Identify categorical vs numeric columns
    cat_cols = []
    for c in cfg.get("cat_cols", []):
        mapped = col_map.get(str(c).strip().lower())
        if mapped is not None and mapped in X.columns:
            cat_cols.append(mapped)
    for c in X.columns:
        if not pd.api.types.is_numeric_dtype(X[c]) and c not in cat_cols:
            cat_cols.append(c)
    num_cols = [c for c in X.columns if c not in cat_cols]

    # Coerce numeric columns and fill NaN
    for c in num_cols:
        X[c] = pd.to_numeric(X[c], errors="coerce")
    if num_cols and X[num_cols].isna().any().any():
        X[num_cols] = X[num_cols].fillna(X[num_cols].median())

    # Encode classification labels to integers
    le = None
    if cfg["task"] == "clf":
        le = LabelEncoder()
        y = pd.Series(le.fit_transform(y), index=y.index)

    # Train / Val / Test split (stratify for clf)
    strat = y if cfg["task"] == "clf" else None
    X_tv, X_test, y_tv, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=SEED, stratify=strat)
    strat_tv = y_tv if cfg["task"] == "clf" else None
    X_train, X_val, y_train, y_val = train_test_split(
        X_tv, y_tv, test_size=VAL_SIZE, random_state=SEED, stratify=strat_tv)

    # Build preprocessor: scale numerics + one-hot categoricals
    transformers = []
    if num_cols:
        transformers.append(("num", StandardScaler(), num_cols))
    if cat_cols:
        transformers.append(("cat", OneHotEncoder(sparse_output=False, handle_unknown="ignore"), cat_cols))
    preprocessor = ColumnTransformer(transformers, remainder="drop")

    # Fit on train only, then transform all splits → pure float32 numpy
    X_train_np = preprocessor.fit_transform(X_train).astype(np.float32)
    X_val_np   = preprocessor.transform(X_val).astype(np.float32)
    X_test_np  = preprocessor.transform(X_test).astype(np.float32)

    y_train_np = y_train.to_numpy(dtype=np.float32)
    y_val_np   = y_val.to_numpy(dtype=np.float32)
    y_test_np  = y_test.to_numpy(dtype=np.float32)

    # Standardize target for regression so RMSE is scale-independent
    y_scaler = None
    if cfg["task"] != "clf":
        y_scaler = StandardScaler()
        y_train_np = y_scaler.fit_transform(y_train_np.reshape(-1, 1)).ravel().astype(np.float32)
        y_val_np   = y_scaler.transform(y_val_np.reshape(-1, 1)).ravel().astype(np.float32)
        y_test_np  = y_scaler.transform(y_test_np.reshape(-1, 1)).ravel().astype(np.float32)

    info = {"n_samples": len(df), "n_num": len(num_cols),
            "n_cat": len(cat_cols), "label_encoder": le, "y_scaler": y_scaler}
    return X_train_np, X_val_np, X_test_np, y_train_np, y_val_np, y_test_np, info

In [22]:
# ========== 4. TRAIN  &  5. EVALUATE  &  6. SAVE ==========

all_results = []
durations = []
t0 = time.time()

for i, ds_id in enumerate(SELECTED, 1):
    cfg = CATALOG[ds_id]

    # ---- ETA estimation ----
    remaining = len(SELECTED) - (i - 1)
    if durations:
        eta_min = np.mean(durations) * remaining / 60
        print(f"[{i}/{len(SELECTED)}] {ds_id}  |  ETA ~ {eta_min:.1f} min")
    else:
        print(f"[{i}/{len(SELECTED)}] {ds_id}  |  estimating ETA after first run...")

    ds_t0 = time.time()

    # ---- Load / Split / Preprocess (returns pure float32 numpy) ----
    X_train, X_val, X_test, y_train, y_val, y_test, info = load_and_split(cfg)

    # ---- Build model ----
    if cfg["task"] == "clf":
        model = XRFM_D_Classifier(random_state=SEED, verbosity=2)
    else:
        model = XRFM_D_Regressor(random_state=SEED, verbosity=2)

    # ---- Train (no cat_col_names — data is already preprocessed) ----
    train_t0 = time.time()
    model.fit(X_train, y_train, X_val, y_val)
    train_sec = time.time() - train_t0

    # ---- Predict ----
    infer_t0 = time.time()
    y_pred = model.predict(X_test)
    infer_ms = (time.time() - infer_t0) / max(len(X_test), 1) * 1000

    # ---- Metrics ----
    metrics = {}
    if cfg["task"] == "clf":
        metrics["accuracy"]  = accuracy_score(y_test, y_pred)
        metrics["f1_macro"]  = f1_score(y_test, y_pred, average="macro")
        try:
            proba = model.predict_proba(X_test)
            if proba.shape[1] == 2:
                metrics["auc"] = roc_auc_score(y_test, proba[:, 1])
            else:
                metrics["auc"] = roc_auc_score(y_test, proba, multi_class="ovr")
        except Exception:
            metrics["auc"] = np.nan
    else:
        metrics["rmse"] = root_mean_squared_error(y_test, y_pred)
        metrics["r2"]   = r2_score(y_test, y_pred)
        # Also report RMSE on the original (un-scaled) target for reference
        ys = info.get("y_scaler")
        if ys is not None:
            y_test_orig = ys.inverse_transform(y_test.reshape(-1, 1)).ravel()
            y_pred_orig = ys.inverse_transform(y_pred.reshape(-1, 1)).ravel()
            metrics["rmse_original"] = root_mean_squared_error(y_test_orig, y_pred_orig)

    # ---- Collect result ----
    row = {
        "dataset": cfg["label"],
        "task": cfg["task"],
        "n_samples": info["n_samples"],
        "n_num": info["n_num"],
        "n_cat": info["n_cat"],
        "train_sec": round(train_sec, 2),
        "infer_ms_per_sample": round(infer_ms, 4),
        **{k: round(v, 4) for k, v in metrics.items()},
    }
    all_results.append(row)

    ds_sec = time.time() - ds_t0
    durations.append(ds_sec)
    print(f"  Done in {ds_sec:.1f}s  |  {metrics}\n")

# ---- Save combined results ----
results_df = pd.DataFrame(all_results)
out_path = OUTPUT_DIR / f"xrfm_results_{RUN_MODE}_{datetime.now():%Y%m%d_%H%M%S}.csv"
results_df.to_csv(out_path, index=False)

print(f"Total: {(time.time()-t0)/60:.1f} min  |  Saved to {out_path}")
display(results_df)

[1/1] online_news_popularity  |  estimating ETA after first run...
Columns classified as continuous: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58]
Columns classified as categorical: []
factory is None, creating factory
kernel_type='l2', M_batch_size=8000
None
Fitting xRFM with 1 trees and 0 iterations per tree


Building trees:   0%|          | 0/1 [00:00<?, ?it/s]

Using top_vector_agop_on_subset split method
Getting AGOP on subset
X_train torch.Size([12887, 59]) y_train torch.Size([12887, 1]) X_val torch.Size([679, 59]) y_val torch.Size([679, 1])
Fitting RFM with ntrain: 12887, d: 59, and nval: 679
Using cheap batch size
Optimal M batch size: 736
Refilling validation set, because at least one split has been made.
Setting categorical indices
Fitting RFM with ntrain: 15223, d: 59, and nval: 3144
Round 0 Val MSE: 0.2704
Sampling AGOP on maximum of 24000 total points


100%|██████████| 2/2 [00:00<00:00, 20.34it/s]

Time taken for round 0: 1.1354591846466064 seconds

...


Building trees: 100%|██████████| 1/1 [00:13<00:00, 13.83s/it]


Final Val MSE: 1.7529
self.best_iter=1


Tuning split temperature: 100%|██████████| 36/36 [00:02<00:00, 13.69it/s]


Selected split_temperature=4.500000000000001 based on validation mse=0.959464
Using soft routing for tree prediction
  Done in 17.3s  |  {'rmse': 0.9195975661277771, 'r2': 0.009348273277282715, 'rmse_original': 10933.6064453125}

Total: 0.3 min  |  Saved to c:\Users\SPY\Desktop\comp9417\group_project\outputs\xrfm_results_single_20260414_170319.csv


,dataset,task,n_samples,n_num,n_cat,train_sec,infer_ms_per_sample,rmse,r2,rmse_original
0,Online News Popularity,reg,39644,59,0,16.51,0.0136,0.9196,0.0093,10933.6064


In [23]:
# ========== Optional: cross-dataset comparison (all mode only) ==========

if RUN_MODE == "all" and len(results_df) > 1:
    cols = [c for c in ["dataset","task","accuracy","f1_macro","auc","rmse","r2","train_sec"]
            if c in results_df.columns]
    display(results_df[cols])
else:
    print("Single mode — no cross-dataset comparison needed.")

Single mode — no cross-dataset comparison needed.
